### **String Manipulation in Python for Bioinformatics**
---
**Who this is for:** 
- You already know basic Python (variables, loops, lists, dictoinaries etc).
- This notebook is a focused refresher on **strings**, using biological examples, so the rest of the repository feels easy to follow.
**Why strings matter in bioinformatics**
---
Almost everything we analyse starts life as text:
- DNA, RNA and protein sequences (`"ATGCGT..."`)
- FASTA headers (`>sample01|Plasmodium falciparum|pfhrp2`)
- FASTQ quality lines (`"IIIIHHGF#"`)
- Gene names, sample IDs and file names
If you can slice, search, clean and transform strings, you can already do a surprising amount of real sequence analysis.
**How each section is organised**
---
1. **Why** you need the tool
2. **How** it works, conceptually
3. **Breakdown** of the code, piece by piece
4. A runnable example
#### **Contents**
---
1. Indexing and slicing
2. Strings are immutable
3. Changing case
4. Counting and searching
5. GC content
6. Replace: DNA to RNA
7. Complement and reverse complement
8. Cleaning with `strip`, `split`, `join`
9. f-strings for readable output
10. Looping over a sequence: codons
11. Finding every position of a motif
12. Validating a sequence
13. Parsing a small FASTA text
14. Bonus: FASTQ quality characters
15. Practice exercises and solutions

### **1. Indexing and slicing**
---
**Why:** You constantly need *part* of a sequence: the first codon, the last 10 bases, a primer region, a gene between two coordinates.

**How it works:** Every character in a string has a position number (an *index*). Python starts counting at **0**, not 1. Negative numbers count from the end.

```
 A  T  G  C  G  T
 0  1  2  3  4  5      <- forward index
-6 -5 -4 -3 -2 -1      <- backward index
```
#### **Breakdown**
----
| Code | Meaning |
|---|---|
| `dna[0]` | the character at position 0 (first) |
| `dna[-1]` | the last character |
| `dna[0:3]` | from position 0 up to, **but not including**, 3 |
| `dna[3:]` | from position 3 to the end |
| `dna[:6]` | from the start up to (not including) 6 |
| `len(dna)` | how many characters |

Remember: in `dna[start:stop]` the **stop** position is excluded.
for more on indexing : https://railsware.com/blog/indexing-and-slicing-for-lists-tuples-strings-sequential-types/

In [ ]:
dna = "ATGCGTACGTTAGC"
print("Length:", len(dna))
print("First base:", dna[0])
print("Last base:", dna[-1])
print("First codon:", dna[0:3])
print("From position 3 onward:", dna[3:])
print("First 6 bases:", dna[:6])

Length: 14
First base: A
Last base: C
First codon: ATG
From position 3 onward: CGTACGTTAGC
First 6 bases: ATGCGT


>**Bioinformatics note:** Biological coordinates (for example in GFF files) often start at **1**, while Python starts at **0**. Off-by-one mistakes between the two are one of the most common bugs you should always watch out for.

#### **2. Strings are immutable**
-----
**Why:** Beginners often try to "fix" one base in a sequence and get an error. Knowing why saves time.

**How it works:** A Python string cannot be changed in place. To "change" it you build a **new** string from pieces of the old one.

**Breakdown**

- `dna[0] = "G"` tries to modify in place, which fails with a `TypeError`.
- `"G" + dna[1:]` builds a new string: the new base `G`, then everything from position 1 onward.

In [ ]:
# let's use a try and excetp block to demonstrate this:
dna = "ATGCGT"
try:
    dna[0] = "G"
except TypeError as error:
    print("Error:", error)

mutated = "G" + dna[1:]
print("Original:", dna)
print("Mutated: ", mutated)

Error: 'str' object does not support item assignment
Original: ATGCGT
Mutated:  GTGCGT


This is exactly how a **point mutation** (a single-base change) is simulated in code.
Don't worry, Biopythong provides another way to make sequence object mutable, we will get there

#### **3. Changing case**
----

**Why:** Sequences can arrive as `atgc`, `ATGC` or mixed. Your comparisons will fail if the case differs (`"A" != "a"`). In some files, lowercase even carries meaning (soft-masked repeats in genome FASTA files).

**How it works:** `.upper()` and `.lower()` return a new string with all letters converted.

**Breakdown**

- `seq.upper()` makes every letter uppercase.
- `seq.lower()` makes every letter lowercase.
- Neither changes `seq` itself (remember: immutable). Save the result in a variable if you want to keep it.

In [ ]:
# let's go through some examples:
messy = "atgCGTacg"
clean = messy.upper()
print(messy)
print(clean)
print(clean.lower())

atgCGTacg
ATGCGTACG
atgcgtacg


#### **4. Counting and searching**
---
**Why:** Questions like "how many A's?", "does it contain a start codon?" and "where is this motif?" are the basics of sequence analysis.

**How it works:** Strings come with built-in methods that scan the text for you.

**Breakdown**

| Code | What it does |
|---|---|
| `seq.count("A")` | how many times `"A"` appears (non-overlapping) |
| `seq.find("ATG")` | position of the **first** match, or `-1` if not found |
| `seq.find("ATG", 1)` | same, but start searching from position 1 |
| `"CGT" in seq` | `True` or `False`: is it present? |
| `seq.startswith("ATG")` | does it begin with this? |
| `seq.endswith("GC")` | does it end with this? |

In [4]:
dna = "ATGCGTACGTTAGCATGC"

print("Number of A:", dna.count("A"))
print("Number of ATG:", dna.count("ATG"))
print("First ATG at position:", dna.find("ATG"))
print("Next ATG after position 1:", dna.find("ATG", 1))
print("Search for something absent:", dna.find("XYZ"))
print("Contains CGT?", "CGT" in dna)
print("Starts with ATG?", dna.startswith("ATG"))
print("Ends with GC?", dna.endswith("GC"))

Number of A: 4
Number of ATG: 2
First ATG at position: 0
Next ATG after position 1: 14
Search for something absent: -1
Contains CGT? True
Starts with ATG? True
Ends with GC? True


#### **5. GC content**
---
**Why:** GC content (the percentage of G and C bases) is a basic property of a sequence. It affects DNA stability, primer design and sequencing quality, and it differs between organisms.
**How it works:** Count the G's and C's, add them, divide by the total length, multiply by 100.
**Breakdown**

1. `dna.count("G") + dna.count("C")` gives the number of G and C bases.
2. `/ len(dna)` turns it into a fraction of the whole.
3. `* 100` turns the fraction into a percentage.

In [5]:
dna = "ATGCGTACGTTAGCATGC"

gc_count = dna.count("G") + dna.count("C")
gc_content = gc_count / len(dna) * 100

print("GC bases:", gc_count)
print("GC content (raw):", gc_content)

GC bases: 9
GC content (raw): 50.0


#### **6. Replace: DNA to RNA**
---
**Why:** Transcription turns a DNA coding sequence into RNA, where **T** becomes **U**.

**How it works:** `.replace(old, new)` swaps every occurrence of `old` with `new` and returns a new string.
**Breakdown:** `dna.replace("T", "U")` means "in `dna`, replace every T with U". So you can build a whole transcription method or function using the Replace method

In [1]:
# let's take an example:
dna = "ATGCGTACGTTAGC"
rna = dna.replace("T", "U")
print("DNA:", dna)
print("RNA:", rna)

DNA: ATGCGTACGTTAGC
RNA: AUGCGUACGUUAGC


#### **7. Complement and reverse complement**
---
**Why:** DNA is double-stranded. A reverse complement gives you the sequence of the **opposite strand**, read in its own 5' to 3' direction. You need it when a gene or primer sits on the reverse strand.

**How it works:** Two steps.

1. **Complement:** swap each base for its partner (A with T, G with C).
2. **Reverse:** flip the whole string end to end.

**Breakdown**
- `str.maketrans("ATGC", "TACG")` builds a translation table: A becomes T, T becomes A, G becomes C, C becomes G.
- `dna.translate(table)` applies that table to every character at once. It is faster and simpler than a chain of `.replace()` calls, which would overwrite their own work (A to T, then T back to A). We could also use a dictionary to create the translation table.
- `[::-1]` is a slice with step `-1`, which reads the string backwards.

In [2]:
# let's work on this example together:
dna = "ATGCGTACGTTAGC"
# the translation table:
table = str.maketrans("ATGC", "TACG")
complement = dna.translate(table) # note that this translation is not the one in protein synthesis:
reverse_complement = complement[::-1] # reverse slicing

# printing output
print("Original:           ", dna)
print("Complement:         ", complement)
print("Reverse complement: ", reverse_complement)

Original:            ATGCGTACGTTAGC
Complement:          TACGCATGCAATCG
Reverse complement:  GCTAACGTACGCAT


> **Why not `.replace()` four times?** Try `dna.replace("A","T").replace("T","A")`. The second step turns the new T's (which were A's) back into A's, so the result is wrong. `translate` swaps all characters in a single pass, which avoids that.

#### **8. Cleaning with `strip`, `split` and `join`**
---

**Why:** Real data is messy: trailing newlines, spaces, headers packed with information separated by symbols. Before analysing anything, you clean and break text apart.

**How it works**

- `strip()` removes whitespace (spaces, tabs, newlines) from both ends.It doesn't remove middle space
- `split(separator)` cuts a string into a **list** of pieces at every separator.
- `join(list)` does the reverse: glues a list of strings into one string.

**Breakdown**

| Code | Result |
|---|---|
| `line.strip()` | removes `\n` and spaces at the ends |
| `header.lstrip(">")` | removes `>` from the **left** end only |
| `header.split("\|")` | splits the text at each `\|` symbol into a list |
| `"-".join(parts)` | glues the pieces together with `-` between them |

In [4]:
# let's work on this example
line = "  ATGCGT\n"
print(repr(line))              # repr shows hidden characters
print(repr(line.strip()))

header = ">sample01|Plasmodium falciparum|pfhrp2"
parts = header.lstrip(">").split("|")
print(parts)
# no that split returns a list, so part is a list of two items, it splits by |.
sample_id = parts[0]
organism = parts[1]
gene = parts[2]
print("Sample:", sample_id)
print("Organism:", organism)
print("Gene:", gene)

print("-".join(parts))

'  ATGCGT\n'
'ATGCGT'
['sample01', 'Plasmodium falciparum', 'pfhrp2']
Sample: sample01
Organism: Plasmodium falciparum
Gene: pfhrp2
sample01-Plasmodium falciparum-pfhrp2


#### **9. f-strings for readable output**
---
**Why:** Printing results with clear labels and sensible decimals makes your work easier to check and share.

**How it works:** Put an `f` before the opening quote, then place variables or expressions inside `{ }`. A format code after a colon controls how a number is shown.

**Breakdown**

| Code | Meaning |
|---|---|
| `f"{name}"` | insert the variable `name` |
| `f"{value:.1f}"` | show a number with **1** decimal place |
| `f"{value:.2f}"` | show a number with **2** decimal places |
| `f"{len(dna)}"` | any expression works inside the braces |

In [5]:
# let's look at this example:
dna = "ATGCGTACGTTAGCATGC"
gc_content = (dna.count("G") + dna.count("C")) / len(dna) * 100

print(f"Sequence length: {len(dna)} bp")
print(f"GC content: {gc_content:.1f}%")
print(f"Number of A: {dna.count('A')}")

Sequence length: 18 bp
GC content: 50.0%
Number of A: 4


#### **10. Looping over a sequence: codons**
---
**Why:** The genetic code is read in **codons**, groups of 3 bases. Splitting a sequence into codons is the first step of translation.

**How it works:** A `for` loop can step through positions. `range(start, stop, step)` with `step = 3` jumps three positions at a time. At each stop we slice out 3 characters.

**Breakdown**
- `range(0, len(dna) - 2, 3)` gives positions 0, 3, 6, ... The `- 2` stops us from slicing an incomplete codon at the end.
- `dna[i:i+3]` takes the 3 characters starting at position `i`.
- `codons.append(...)` adds each codon to a list.

In [6]:
# let's work this example:
dna = "ATGCGTACGTTAGCATGC"

codons = []
for i in range(0, len(dna) - 2, 3):
    codons.append(dna[i:i+3])

print(codons)
print("Number of codons:", len(codons))

['ATG', 'CGT', 'ACG', 'TTA', 'GCA', 'TGC']
Number of codons: 6


You can also loop directly over characters, for example to count purines (A and G) by hand:

In [7]:
# we set a counter value and anytime we come a cross A or T we increment out counter value
purines = 0
for base in dna:
    if base == "A" or base == "G":
        purines += 1

print("Purines (A + G):", purines)

Purines (A + G): 9


#### **11. Finding every position of a motif**
---
**Why:** `find()` returns only the first match. To locate **all** occurrences (restriction sites, start codons, primer binding sites) you repeat the search from just after the previous hit.

**How it works:** Keep searching with `find`, each time starting one position after the last match. When `find` returns `-1`, there are no more matches.

**Breakdown**

1. `start = dna.find(motif)` finds the first hit.
2. `while start != -1:` repeats as long as a hit was found.
3. Inside the loop, save the position, then search again from `start + 1`.

In [8]:
# Here is an example for you to understand what is going on:
dna = "ATGCGTACGTTAGCATGC"
motif = "ATG"

positions = []
start = dna.find(motif)
while start != -1:
    positions.append(start)
    start = dna.find(motif, start + 1)

print(f"{motif} found at positions (0-based): {positions}")

ATG found at positions (0-based): [0, 14]


#### **12. Validating a sequence**
---
**Why:** A typo, a space or a stray number in a sequence can silently break an analysis. Checking that a sequence contains only valid bases is a cheap safety step.

**How it works:** A `set` keeps only the unique characters. If the unique characters in your sequence are all inside the allowed set, the sequence is valid. There many ways of validating sequence and along the line, you will see me use different validation method in some of the notebooks. These process are just for you to understand concepts better. There a finer, efficient tools already doing most of what we have covered.

**Breakdown**

- `set(seq)` gives the unique characters, for example `{"A", "T", "G"}`.
- `set("ATGC")` is the allowed alphabet.
- `<=` means "is a subset of": every character in `seq` must also be in the allowed alphabet.
- `.upper()` first, so lowercase input still passes.

In [9]:
# let's try an example:
good = "ATGCGTACG"
bad = "ATGCXTACG"
allowed = set("ATGC")

print("good valid?", set(good.upper()) <= allowed)
print("bad valid?", set(bad.upper()) <= allowed)

# Which characters are the problem?
print("Invalid characters in bad:", set(bad.upper()) - allowed)

good valid? True
bad valid? False
Invalid characters in bad: {'X'}


#### **13. Parsing a small FASTA text**
---

**Why:** FASTA is the most common sequence file format. A record has a header line starting with `>`, followed by one or more lines of sequence. Parsing it by hand shows you what libraries like Biopython do for you.

**How it works:** Read line by line. A line starting with `>` begins a new record, so we store its name. Any other line is sequence and gets added to the current record.

**Breakdown**

- `fasta_text.strip().split("\n")` cleans the text and breaks it into a list of lines.
- `line.startswith(">")` detects a header.
- `line[1:].split()[0]` drops the `>` and keeps only the first word as the record name.
- `records[name] += line.strip()` appends cleaned sequence text to that record.

In [12]:
# let's look at this example:
fasta_text = """>seq1 sample A
ATGCGTACG
TTAGCATGC
>seq2 sample B
GGGCCCATA
TTTAAA
"""
# here is the logic
records = {}
for line in fasta_text.strip().split("\n"):
    if line.startswith(">"):
        name = line[1:].split()[0]
        records[name] = ""
    else:
        records[name] += line.strip()
# let's print what we collected from the loop:
for name, seq in records.items():
    print(f"{name}: {seq} ({len(seq)} bp)")

seq1: ATGCGTACGTTAGCATGC (18 bp)
seq2: GGGCCCATATTTAAA (15 bp)


> Later, you will meet `SeqIO.parse()` in Biopython, which does this job (and handles many edge cases) in one line. Understanding the manual version means you will know what it is doing underneath.

#### **14.FASTQ quality characters**
---

**Why:** In a FASTQ file, every base has a quality character. These characters are how sequencing machines say how confident they are when calling out each base. To filter low-quality data, you convert them to numbers.

**How it works:** Each character is secretly a number (its ASCII code). `ord(character)` gives that number. In the standard modern encoding, the **Phred quality score** is `ord(character) - 33`. A higher score means higher confidence.

**Breakdown**

- `ord("I")` returns `73`.
- `73 - 33 = 40`, a Phred score of 40, meaning about a 1 in 10,000 chance the base call is wrong.
- `ord("#")` returns `35`, so the score is `2`, which is very poor.

In [ ]:
# let's look at this example to make things clearer.
quality = "IIIIHHGF#"
scores = []
for character in quality:
    scores.append(ord(character) - 33)

print(scores)
print("Average quality:", sum(scores) / len(scores))

[40, 40, 40, 40, 39, 39, 38, 37, 2]
Average quality: 35.0


#### **15.Try your hands on these tasks**
---
Try each one yourself in the empty cell, then compare with the solutions below.

1. For `seq = "ATGCTTAGTT"`, count how many `T` bases there are and print the result as a percentage of the sequence length with 1 decimal.
2. Find the **reverse complement** of `"GATTACA"`.
3. Convert `"ATGGCCTAA"` to RNA.
4. From the header `"gene=pfkelch13;organism=P. falciparum;length=726"`, extract just the gene name (`pfkelch13`).
5. Check whether `"ATGAAATAG"` starts with a start codon (`ATG`) and ends with a stop codon (`TAA`, `TAG` or `TGA`).
6. Find all positions of `"GC"` in `"ATGCGTAGCGC"`.

In [16]:
# Exercise 1: write your code here

In [17]:
# Exercise 2: write your code here

In [18]:
# Exercise 3: write your code here

In [19]:
# Exercise 4: write your code here

In [20]:
# Exercise 5: write your code here

In [21]:
# Exercise 6: write your code here

#### **Solutions**
---
Look only after you have tried.

In [22]:
# Solution 1
seq = "ATGCTTAGTT"
t_percent = seq.count("T") / len(seq) * 100
print(f"T content: {t_percent:.1f}%")

T content: 50.0%


In [23]:
# Solution 2
seq = "GATTACA"
table = str.maketrans("ATGC", "TACG")
print(seq.translate(table)[::-1])

TGTAATC


In [24]:
# Solution 3
print("ATGGCCTAA".replace("T", "U"))

AUGGCCUAA


In [25]:
# Solution 4
header = "gene=pfkelch13;organism=P. falciparum;length=726"
first_field = header.split(";")[0]      # "gene=pfkelch13"
gene = first_field.split("=")[1]        # "pfkelch13"
print(gene)

pfkelch13


In [26]:
# Solution 5
seq = "ATGAAATAG"
starts_ok = seq.startswith("ATG")
ends_ok = seq.endswith(("TAA", "TAG", "TGA"))   # endswith accepts a tuple
print("Starts with ATG:", starts_ok)
print("Ends with a stop codon:", ends_ok)

Starts with ATG: True
Ends with a stop codon: True


In [27]:
# Solution 6
seq = "ATGCGTAGCGC"
positions = []
start = seq.find("GC")
while start != -1:
    positions.append(start)
    start = seq.find("GC", start + 1)
print(positions)

[2, 7, 9]


#### **What we will do next:**
---

Everything you did by hand here has a ready-made equivalent in **Biopython**:
| What you did | Biopython equivalent |
|---|---|
| reverse complement with `translate` and `[::-1]` | `Seq.reverse_complement()` |
| DNA to RNA with `replace` | `Seq.transcribe()` |
| codons plus a lookup table | `Seq.translate()` |
| manual FASTA parsing | `SeqIO.parse()` |

Knowing the manual version means you can tell when a tool is doing something unexpected.Note that production tools use efficient algorithms which an advancement of some of these basic algorithms.